# AI Mobile Network Optimization — Simple Learning Notebook

This notebook uses a **real CSV in this folder** (no data is generated here).

**How to run:** click the first code cell, then use **Run All**.

**For trainers (no coding skill needed):** above every code cell there is one **Ask AI** line. Copy that line, paste it to an AI, and it should write the same code.

---

## End-to-end path (beginner → finished)

Each box is one step. Follow the arrows **left to right**.

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│ 1. Load data │ ──► │ 2. Understand│ ──► │ 3. Check data│ ──► │ 4. Explore   │ ──► │ 5. Prepare   │
│    open CSV  │     │    columns   │     │  missing/dups│     │  QoS+coverage│     │  features    │
└──────────────┘     └──────────────┘     └──────────────┘     └──────────────┘     └──────┬───────┘
                                                                                           │
                                                                                           ▼
┌──────────────┐     ┌──────────────┐     ┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│ 6. Split     │ ──► │ 7. Train     │ ──► │ 8. Score     │ ──► │ 9. Show      │ ──► │ 10. Act      │
│ past / future│     │    model     │     │  R² / MAE    │     │   results    │     │ recommend fix│
└──────────────┘     └──────────────┘     └──────────────┘     └──────────────┘     └──────────────┘
```

Row 1 = get the data ready. Row 2 = train the model and use the result.

### Machine-learning steps

```
┌─────────┐     ┌─────────┐     ┌─────────┐     ┌─────────┐     ┌──────────────┐     ┌──────────────┐
│ Prepare │ ──► │  Split  │ ──► │  Train  │ ──► │  Score  │ ──► │ Show results │ ──► │ Recommend fix│
└─────────┘     └─────────┘     └─────────┘     └─────────┘     └──────────────┘     └──────────────┘
```

### What you will learn

1. **Performance monitoring + QoS** — which cells are slow, laggy, or dropping calls
2. **Coverage gap assessment** — which cells have weak signal (RSRP)
3. **Predict the future with ML** — learn from morning/afternoon, forecast the evening
4. **Optimization and planning** — turn those forecasts into a recommended fix


---
## 1. Load the dataset

Think of a **mobile network** like a city of Wi‑Fi routers — except each “router” is a **cell tower** that phones connect to.

The file `network_optimization_dataset.csv` is a **diary of those towers**.

- **720 rows** = 5 regions × 6 towers × 24 hours
- **One row** = *what happened at one tower during one hour*

Example: “At 7pm in Addis Ababa, tower ADD-01 had 41 people on it, the signal was OK, and a few calls dropped.”

**This cell only needs `pandas`.** Charts and machine-learning libraries are imported later, in the cells that use them.


**Ask AI (copy this):** Load network_optimization_dataset.csv with pandas only, print how many rows and columns, and show the first 5 rows.

In [ ]:
# pandas = open and work with tables (like Excel, but in Python)
# We only import pandas here. Charts and machine learning come in later cells.
import pandas as pd

df = pd.read_csv("network_optimization_dataset.csv")
print("Loaded: network_optimization_dataset.csv")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
df.head()


### Dataset in plain English (no telecom background needed)

Imagine each tower is a **small shop**.

- **Capacity** = how many customers the shop can serve
- **Users / traffic** = how many customers actually came
- **Utilization** = how full the shop is (`80%` means it is getting crowded)
- **Throughput** = how fast the internet feels (download speed)
- **Latency** = waiting time before a page or call starts (`low` is good)
- **Drop rate** = phone calls that die in the middle
- **Packet loss** = missing pieces of the message (choppy video / broken voice)
- **RSRP** = how strong the signal bars are (`-70` is strong, `-110` is almost no signal)
- **SINR** = how *clear* the signal is vs noise (like hearing someone in a quiet room vs a loud market)
- **Coverage status** = is this area covered? (`Gap` = a dead zone)
- **QoS class** = overall grade for the user experience (`Excellent` → `Poor`)

**The three questions this table answers**

1. **Is the service good?** → look at QoS columns (`qos_class`, drop rate, latency, speed)
2. **Can phones even hear the tower?** → look at coverage columns (`rsrp_dbm`, `coverage_status`)
3. **Should we upgrade this tower?** → look at how full it is (`utilization_pct`) plus the two points above


**Column-by-column cheat sheet**

| Column | Everyday meaning | What “good” looks like |
|---|---|---|
| `cell_id` | Tower name (e.g. ADD-01) | — |
| `region` | City / region of that tower | — |
| `hour` | Hour of the day (0 = midnight, 19 = 7pm) | Evening is usually busiest |
| `active_users` | How many phones were connected | Depends on the area |
| `traffic_mbps` | How much data people used | High at peak hours |
| `capacity_mbps` | Maximum data the tower can handle | Bigger = can serve more people |
| `utilization_pct` | How full the tower is (used ÷ max) | Below ~70% is comfortable |
| `throughput_mbps` | Speed the user actually gets | Higher is better |
| `latency_ms` | Delay in milliseconds | Lower is better (20 ms feels snappy, 80+ feels laggy) |
| `drop_rate_pct` | % of calls that failed | Near 0% is good |
| `packet_loss_pct` | % of data that never arrived | Near 0% is good |
| `rsrp_dbm` | Signal strength (the “bars”) | `-70` strong · `-110` almost dead |
| `sinr_db` | How clean the signal is | Higher is clearer |
| `coverage_status` | Coverage grade | Good / Fair / Poor / **Gap** (dead zone) |
| `qos_class` | User-experience grade | Excellent / Good / Fair / Poor |
| `distance_km` | How far the typical user is from the tower | Closer usually means stronger signal |

**Ask AI (copy this):** Take the first row of df and print it in everyday sentences: time, tower, city, how many people, how full the tower is, speed, wait time, dropped calls, signal, and the QoS grade.

In [ ]:
# Pick one real row and say it in everyday language
row = df.iloc[0]

print("One row from the file, translated:\n")
print(
    f"At {int(row['hour']):02d}:00, tower {row['cell_id']} in {row['region']} "
    f"had {int(row['active_users'])} people connected."
)
print(
    f"They used {row['traffic_mbps']} Mbps of data. The tower can handle "
    f"{int(row['capacity_mbps'])} Mbps, so it was {row['utilization_pct']}% full "
    f"(like a shop that is {row['utilization_pct']}% busy)."
)
print(
    f"Internet speed was {row['throughput_mbps']} Mbps, wait time was "
    f"{row['latency_ms']} ms, and {row['drop_rate_pct']}% of calls dropped."
)
print(
    f"Signal strength (RSRP) was {row['rsrp_dbm']} dBm — coverage = {row['coverage_status']}."
)
print(f"Overall user experience grade (QoS) = {row['qos_class']}.")
print(f"People were about {row['distance_km']} km from the tower.")

---
## 2. Quick look at the data

Always start with shape, missing values, and a summary. This is the same habit for any KPI file.


**Ask AI (copy this):** Using the table df, print how many missing values, how many duplicate rows, how many regions and towers, then show a summary of the numbers.

In [ ]:
print("Missing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Regions:", df["region"].nunique(), "| Cells:", df["cell_id"].nunique())
print()
df.describe().round(2)


---
## 3. Performance monitoring + QoS

QoS answers: *is the user experience good right now?*

We look at:
- traffic by hour (when the network is busy)
- how many hours fall into each QoS class
- latency vs throughput (busy cells are usually slower)


**Ask AI (copy this):** Draw two charts: average active users by hour as a line, and a bar chart of how many hours are Excellent, Good, Fair, Poor; also print average QoS numbers by region.

In [ ]:
# matplotlib = draw charts. Import it here — this is the first chart cell.
import matplotlib.pyplot as plt

hourly = df.groupby("hour")[["active_users", "drop_rate_pct", "latency_ms"]].mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(hourly.index, hourly["active_users"], marker="o")
axes[0].set_title("Average users by hour")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Active users")
axes[0].grid(alpha=0.3)

qos_order = ["Excellent", "Good", "Fair", "Poor"]
qos_counts = df["qos_class"].value_counts().reindex(qos_order)
axes[1].bar(qos_counts.index, qos_counts.values)
axes[1].set_title("Hours in each QoS class")
axes[1].set_ylabel("Number of rows")

plt.tight_layout()
plt.show()

print("Average QoS by region:")
df.groupby("region")[["throughput_mbps", "latency_ms", "drop_rate_pct", "packet_loss_pct"]].mean().round(2)


**Ask AI (copy this):** Scatter plot throughput vs latency colored by drop rate, titled speed vs delay, then show the 8 worst hours with the highest drop rate.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.scatter(df["throughput_mbps"], df["latency_ms"], c=df["drop_rate_pct"], alpha=0.35)
plt.colorbar(label="Drop rate (%)")
plt.xlabel("Throughput (Mbps)")
plt.ylabel("Latency (ms)")
plt.title("QoS picture: speed vs delay (color = dropped calls)")
plt.grid(alpha=0.3)
plt.show()

print("Worst 8 hours (highest drop rate):")
df.nlargest(8, "drop_rate_pct")[
    ["cell_id", "region", "hour", "drop_rate_pct", "latency_ms", "throughput_mbps", "qos_class"]
]


---
## 4. Coverage gap assessment

A **coverage gap** is a place where the radio signal is too weak.

Simple rule used in this dataset:

| RSRP | Status |
|---|---|
| ≥ -85 dBm | Good |
| -100 to -85 | Fair |
| -110 to -100 | Poor |
| < -110 dBm | **Gap** |


**Ask AI (copy this):** Count coverage_status, then for each tower find average signal and how many Gap hours; call it a coverage gap if Gap hours are 8 or more; plot RSRP with a line at -110 and signal vs distance; list problem towers.

In [ ]:
import matplotlib.pyplot as plt

print("Coverage mix:")
print(df["coverage_status"].value_counts(), "\n")

# One number per cell (average over 24 hours)
cell_cover = (
    df.groupby(["cell_id", "region"])
    .agg(
        avg_rsrp=("rsrp_dbm", "mean"),
        avg_sinr=("sinr_db", "mean"),
        distance_km=("distance_km", "mean"),
        gap_hours=("coverage_status", lambda s: (s == "Gap").sum()),
        poor_hours=("coverage_status", lambda s: (s == "Poor").sum()),
    )
    .reset_index()
)

def coverage_label(row):
    if row["gap_hours"] >= 8:
        return "Coverage gap"
    if row["poor_hours"] + row["gap_hours"] >= 8:
        return "Weak coverage"
    if row["avg_rsrp"] < -85:
        return "Fair — watch"
    return "OK"

cell_cover["coverage_verdict"] = cell_cover.apply(coverage_label, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["rsrp_dbm"], bins=20)
axes[0].axvline(-110, linestyle="--", label="Gap threshold (-110 dBm)")
axes[0].set_title("RSRP distribution")
axes[0].set_xlabel("RSRP (dBm)")
axes[0].legend()

axes[1].scatter(cell_cover["distance_km"], cell_cover["avg_rsrp"], alpha=0.8)
axes[1].axhline(-110, linestyle="--")
axes[1].set_title("Farther cells usually have weaker signal")
axes[1].set_xlabel("Distance to tower (km)")
axes[1].set_ylabel("Average RSRP (dBm)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Cells with a coverage problem:")
cell_cover[cell_cover["coverage_verdict"] != "OK"].sort_values("avg_rsrp")


---
## 5. Predict QoS risk (simple AI model)

**Target:** `drop_rate_pct` — a clear QoS problem signal.

**Features:** things we can know *before* the drop happens (load, capacity, signal, hour).

We do **not** use throughput, latency, or packet loss as inputs — those are already symptoms of the same problem.

This first model is a **practice test** (random rows). The next section is the real idea: train on the **past**, predict the **future**.


**Ask AI (copy this):** Train a Random Forest to predict drop_rate_pct from hour, users, traffic, capacity, signal, and distance; use an 80/20 split; print R² and MAE; plot predicted vs actual and which inputs matter most.

In [ ]:
# sklearn = machine learning. Import only what this cell uses.
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt

df["is_peak_hour"] = df["hour"].isin([8, 9, 18, 19, 20]).astype(int)

df_model = pd.get_dummies(df, columns=["region"], drop_first=True)
region_cols = [c for c in df_model.columns if c.startswith("region_")]

feature_cols = [
    "hour", "is_peak_hour", "active_users", "traffic_mbps",
    "capacity_mbps", "rsrp_dbm", "sinr_db", "distance_km",
] + region_cols

X = df_model[feature_cols]
y = df_model["drop_rate_pct"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=150, max_depth=8, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print(f"R²  = {r2_score(y_test, pred):.3f}   (1.0 = perfect)")
print(f"MAE = {mean_absolute_error(y_test, pred):.3f} percentage points")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(y_test, pred, alpha=0.4)
lims = [0, max(y_test.max(), pred.max())]
axes[0].plot(lims, lims, "--", color="gray")
axes[0].set_xlabel("Actual drop rate (%)")
axes[0].set_ylabel("Predicted drop rate (%)")
axes[0].set_title("Predicted vs actual")

imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
axes[1].barh(imp.index, imp.values)
axes[1].set_title("What drives predicted drop rate?")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()


---
## 5b. Predict the future (learn from the past)

Machine learning does **not** see the future. It learns a pattern from old data, then guesses what comes next — like a weather forecast.

**Our split (simple and honest):**

| Data | Hours | Meaning |
|---|---|---|
| Train (the past) | 0–17 | morning + afternoon — the model may study this |
| Test (the future) | 18–23 | evening — the model must **guess**, we check if it was right |

We only give the model things we would already know earlier in the day:
tower size, typical signal, typical users, region, and the hour we want to forecast.

Then we try two models:

1. **Linear Regression** — a straight-line guess (simple)
2. **Random Forest** — many small decision trees (usually better)

We also train a third model that answers a yes/no question: *will this evening hour be Poor QoS?*

**Ask AI (copy this):** Train on hours 0 to 17 and forecast hours 18 to 23: compare Linear Regression and Random Forest for evening drop rate, also predict if QoS will be Poor, then plot predicted vs actual drop rate by evening hour.

In [ ]:
# New models for this step only: straight line, forest, and yes/no classifier
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score
import matplotlib.pyplot as plt

# Past = hours 0-17. Future = evening 18-23 (the model has not seen this yet)
past = df_model[df_model["hour"] <= 17].copy()
future = df_model[df_model["hour"] >= 18].copy()

# What we already know about each tower from earlier today
known = (
    past.groupby("cell_id")[["rsrp_dbm", "sinr_db", "active_users"]]
    .mean()
    .rename(columns={
        "rsrp_dbm": "typical_rsrp",
        "sinr_db": "typical_sinr",
        "active_users": "typical_users",
    })
)
past = past.merge(known, on="cell_id")
future = future.merge(known, on="cell_id")

forecast_cols = [
    "hour", "is_peak_hour", "capacity_mbps", "distance_km",
    "typical_rsrp", "typical_sinr", "typical_users",
] + region_cols

X_past, y_past = past[forecast_cols], past["drop_rate_pct"]
X_future, y_future = future[forecast_cols], future["drop_rate_pct"]

linear = LinearRegression()
linear.fit(X_past, y_past)
pred_linear = linear.predict(X_future)

forest = RandomForestRegressor(n_estimators=150, max_depth=8, random_state=42)
forest.fit(X_past, y_past)
pred_forest = forest.predict(X_future)

print("Forecast evening drop rate (trained on hours 0–17, tested on 18–23)\n")
print(f"Linear Regression   R²={r2_score(y_future, pred_linear):.3f}  MAE={mean_absolute_error(y_future, pred_linear):.3f}")
print(f"Random Forest       R²={r2_score(y_future, pred_forest):.3f}  MAE={mean_absolute_error(y_future, pred_forest):.3f}")

# Will tonight's hour be Poor QoS? (yes/no)
y_past_poor = (past["qos_class"] == "Poor").astype(int)
y_future_poor = (future["qos_class"] == "Poor").astype(int)
clf = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
clf.fit(X_past, y_past_poor)
poor_pred = clf.predict(X_future)
print(f"\nPoor-QoS classifier accuracy on evening hours: {accuracy_score(y_future_poor, poor_pred):.1%}")

future = future.copy()
future["forecast_drop"] = pred_forest
future["forecast_poor"] = poor_pred

by_hour = future.groupby("hour")[["drop_rate_pct", "forecast_drop"]].mean()

plt.figure(figsize=(8, 4))
plt.plot(by_hour.index, by_hour["drop_rate_pct"], marker="o", label="What actually happened")
plt.plot(by_hour.index, by_hour["forecast_drop"], marker="s", label="Model forecast (made from earlier hours)")
plt.title("Evening forecast: predicted vs actual drop rate")
plt.xlabel("Hour")
plt.ylabel("Average drop rate (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Ask AI (copy this):** From the evening forecast, make a table per tower with predicted drop rate, actual drop rate, and hours forecast as Poor, mark High risk if predicted drop is over 1.5, and show the top 10 towers.

In [ ]:
# Tonight's risk list: which towers are forecast to have trouble this evening?
cell_region = df[["cell_id", "region"]].drop_duplicates()

tonight = (
    future.groupby("cell_id")
    .agg(
        forecast_drop=("forecast_drop", "mean"),
        actual_drop=("drop_rate_pct", "mean"),
        hours_forecast_poor=("forecast_poor", "sum"),
    )
    .reset_index()
    .merge(cell_region, on="cell_id")
    .sort_values("forecast_drop", ascending=False)
)

tonight["risk"] = "Lower"
tonight.loc[tonight["forecast_drop"] > 1.5, "risk"] = "High — act before evening"

print("Evening forecast by tower (highest predicted drop first):\n")
tonight[["cell_id", "region", "forecast_drop", "actual_drop", "hours_forecast_poor", "risk"]].head(10)

---
## 6. Optimization recommendations

A prediction is not a plan. Combine **predicted drop rate** with **coverage** and **utilization** to choose an action:

- weak signal → coverage fix (tilt, relay, or new site)
- high load → add capacity / offload
- both → highest priority


**Ask AI (copy this):** Predict drop rate for every row, keep busy peak hours, and for each tower recommend an action: fix coverage if signal is weak, add capacity if the tower is too full, do both if it is high risk and weak and busy, then save the list as network_optimization_recommendations.csv.

In [ ]:
df["predicted_drop_rate"] = model.predict(df_model[feature_cols])

# Use the busy evening hours — that is when users feel problems most
busy = df[df["is_peak_hour"] == 1]

plan = (
    busy.groupby(["cell_id", "region"])
    .agg(
        predicted_drop=("predicted_drop_rate", "mean"),
        utilization_pct=("utilization_pct", "mean"),
        rsrp_dbm=("rsrp_dbm", "mean"),
        distance_km=("distance_km", "mean"),
        capacity_mbps=("capacity_mbps", "mean"),
        latency_ms=("latency_ms", "mean"),
    )
    .reset_index()
)

def recommend(row):
    weak = row["rsrp_dbm"] < -100
    busy_cell = row["utilization_pct"] > 80
    risky = row["predicted_drop"] > 1.5

    if risky and weak and busy_cell:
        return "High priority: add capacity AND fix coverage"
    if risky and weak:
        return "Fix coverage (tilt / relay / new site)"
    if risky and busy_cell:
        return "Add capacity or offload to a neighbor cell"
    if weak:
        return "Investigate coverage"
    if row["predicted_drop"] > 0.8:
        return "Monitor"
    return "No action"

plan["action"] = plan.apply(recommend, axis=1)
plan = plan.sort_values("predicted_drop", ascending=False)

plan.to_csv("network_optimization_recommendations.csv", index=False)
print("Saved: network_optimization_recommendations.csv")
print()
print(plan["action"].value_counts().to_string())
print()
plan.head(10)


---
## 7. What-if: test a fix before spending money

Take the **highest-priority cell**. If we increase its capacity by 50%, does predicted drop rate go down?


**Ask AI (copy this):** Take the first tower in the plan table, predict its average drop rate, then predict again after increasing capacity by 50%, print both numbers, and draw a before/after bar chart.

In [ ]:
import matplotlib.pyplot as plt

top = plan.iloc[0]["cell_id"]
scene = df_model[df_model["cell_id"] == top].copy()

before = model.predict(scene[feature_cols]).mean()

after_scene = scene.copy()
after_scene["capacity_mbps"] = after_scene["capacity_mbps"] * 1.5
after = model.predict(after_scene[feature_cols]).mean()

print(f"Cell: {top}")
print(f"Predicted drop rate BEFORE: {before:.2f}%")
print(f"Predicted drop rate AFTER +50% capacity: {after:.2f}%")

plt.figure(figsize=(5, 4))
plt.bar(["Before", "After +50% capacity"], [before, after])
plt.ylabel("Predicted drop rate (%)")
plt.title(f"Capacity upgrade test — {top}")
plt.show()


---
## What you practiced

1. Loaded a KPI file and checked it
2. Monitored **QoS** and found **coverage gaps**
3. Trained models to **forecast the evening** from earlier hours (Linear Regression + Random Forest)
4. Predicted **Poor QoS** as a yes/no question
5. Turned forecasts into a **planning list** and simulated a fix

**Important:** the model is guessing from patterns, not magic. It will be wrong sometimes. That is why we always compare the forecast with what actually happened.

**Try next**
- Change the cut: train on hours 0–15, forecast 16–23
- Forecast `latency_ms` instead of drop rate
- Change the High-risk threshold from 1.5 to 1.0 and see which towers appear
